# RR Clutch Index: reproducible notebook

This notebook regenerates every final table in `outputs/tables/` from the raw ball-by-ball file. It runs the same scripts as the pipeline (`src/`), with fixed random seed **2027**.

- **Raw data:** `data/raw/ball_by_ball_v2.csv.gz`. Its checksum is verified before anything runs.
- **Method rules:** `logs/PREREGISTRATION.md` (frozen) and `logs/DECISIONS.md`.
- **Round 1 question:** who are the top 10 batters and the top 10 bowlers who perform best under pressure? (Organisers' clarification, D38.)
- **What the score is:** a *historical pressure-performance indicator* for IPL 2023–26. It is not a forecast (D15).

Runtime is about 3–4 minutes.

In [1]:
import sys, os, json, runpy
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src' / 'config.py').exists():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
from config import SEED, RAW_SHA256
print('repo:', ROOT, '| seed:', SEED)

repo: /home/user/RR_Expert | seed: 2027


## 1. Verify the raw data
The loader refuses to run if the checksum or shape differs from the validated v2 dataset.

In [2]:
from io_raw import load_raw
raw = load_raw()
print(raw.shape, 'sha256 OK:', RAW_SHA256[:16], '...')

(200999, 34) sha256 OK: 01e2b4379edfbab6 ...


## 2. Run the whole pipeline
The steps run in order: Step 0 audit and official match-list check → win probability → pressure → expected runs → metrics and score → scope sensitivity → payload.
(The Step 5 auction tables are still produced for the record but are not part of the Round 1 answer.)
Facts are re-read from saved evidence files. There is no web access.

In [3]:
import run_all, time
t0 = time.time()
run_all.main()
print(f'pipeline finished in {time.time() - t0:.0f}s')

=== step0_match_keys.py


367 IPL matches; {2022: 74, 2023: 74, 2024: 71, 2025: 74, 2026: 74}
duplicate (season, date, team pair): 0
first-ball clock range: 10:00:13 - 20:31:28
=== step0_match_keys.py done in 2s


=== step0_audit.py


IPL matches 367 / all 867; IPL balls 87687 / all 200999; last IPL date 2026-05-31
matches per season: {2022: 74, 2023: 74, 2024: 71, 2025: 74, 2026: 74}
result categories: {'CHASING_SIDE_REACHED_TARGET': 186, 'DEFENDING_SIDE_WON': 167, 'UNDETERMINED_FROM_DATA': 12, 'TIE_DECIDED_BY_SUPER_OVER': 2}
reduced/DLS suspected: 13 | no-result suspected: 5 | wp eligible: 353
super over matches: [40725, 50359] | players: 382 | ID check rows: 87
reconciliation OK: True | clock claim: True
done
=== step0_audit.py done in 18s


=== step0_finalize.py


eligible: 353 of 367 | by season: {2022: 72, 2023: 71, 2024: 70, 2025: 69, 2026: 71}
facts-reported interrupted matches all inside data-flagged set: [20226, 21586, 40727, 40737, 40748, 40749, 40751, 50136, 50137]
playoff ids agree with facts: [15187, 15188, 15189, 15190, 21583, 21584, 21585, 21586, 31289, 31290, 31291, 31292, 40764, 40765, 40766, 40767, 50942, 50943, 50944, 50945]
=== step0_finalize.py done in 0s


... (21 outputs left out)

gaps:
                              area  gap_vs_league_per100  se_per100                         verdict  rr_balls
   Batting under pressure: Middle                  -7.3       11.3 within noise (directional only)       243
    Batting under pressure: Death                  -3.2       10.8 within noise (directional only)       359
   Bowling under pressure: Middle                   1.8       10.0 within noise (directional only)       310
    Bowling under pressure: Death                   5.7       11.8 within noise (directional only)       256
Bowling under pressure: Powerplay                   7.1       13.1 within noise (directional only)       199
Batting under pressure: Powerplay                  52.9       15.4                    above league       228
slots 5 overseas slots 1 | pool 55 of 110 scorable
      tier         player_name                      sub_role  score  score_lo80  score_hi80  ipl2026_balls                     fills_gaps
 Must-Back     Ashutosh Sharma           

=== step7_payload.py


payload bytes: 920427 | players: 257 | showcase: 8 | agg rows: 5292
{
"seasons_scored": "2023\u20132026",
"training_only": "2022",
"ipl_matches": 367,
"ipl_balls": 87687,
"ipl_matches_span": "2022\u20132026",
"scored_matches": 293,
"scored_balls": 67880,
"wp_eligible": 353,
"last_match": "2026-05-31",
"in_scope": 257,
"scorable_bat": 47,
"scorable_bowl": 71,
"scorable_ar": 8,
"not_scorable": 147,
"brier": {
"2023": 0.217,
"2024": 0.166,
"2025": 0.169,
"2026": 0.183
},
"brier_inn": {
"2023_1": 0.248,
"2023_2": 0.185,
"2024_1": 0.205,
"2024_2": 0.125,
"2025_1": 0.205,
"2025_2": 0.13,
"2026_1": 0.24,
"2026_2": 0.121
},
"high_share": {
"2023": 22.8,
"2024": 14.3,
"2025": 22.4,
"2026": 19.9
},
"backtest": {
"bat_score": 42.9,
"bat_naive": 46.7,
"bat_diff_lo": -17.1,
"bat_diff_hi": 3.8,
"bowl_score": 72.8,
"bowl_naive": 75.0,
"bowl_diff_lo": -14.6,
"bowl_diff_hi": 4.4,
"bat_rho": -0.13,
"bat_rho_naive": -0.15,
"bowl_rho": -0.05,
"bowl_rho_naive": 0.29,
"bat_base": 50.8,
"bowl_base": 59.5,
"b

pipeline finished in 128s


## 3a. Our IPL match list against the organisers' official match file
Every one of our matches is an official IPL match with the same season and date; playoffs, no-results and super overs agree. The 7 rain-affected matches where the ball data alone cannot name a winner are all left out of the win-probability model.

In [4]:
display(pd.read_csv(ROOT / 'outputs' / 'tables' / 'step0_official_meta_check.csv'))

,check,status,detail
0,One IPL competition per season 2022-2026,Pass,"comp_ids 2022:1358, 2023:1715, 2024:2193, 2025..."
1,Every match in our IPL table is in the officia...,Pass,0 of 367 not found
2,Season agrees,Pass,0 differ
3,Date agrees,Pass,0 differ
4,Official matches absent from the ball data are...,Pass,"4 absent: 31281 (2024, Match Abandoned); 31284..."
5,Winner agrees (no result = no winner),Differs,7 of 367 differ
6,Playoff flag agrees with official stage,Pass,20 official playoff matches; 0 differ
7,Reduced-overs (DLS) matches agree,Differs,"official 5, ours 13; 8 differ"
8,No-result matches agree,Pass,"official 5, ours 5; 0 differ"
9,Super-over matches agree,Pass,"official 2, ours 2; 0 differ"


## 3. Independent sanity check: official season leaders, recomputed from raw
Part 5 definitions: batter runs are runs off the bat; bowler wickets exclude run-outs; super overs are excluded.

In [5]:
ipl = {'1','2','3','4','5','7','8','242','3508','3509'}
m = raw.groupby('match_id').apply(lambda g: set(g.batting_team_id) | set(g.bowling_team_id) <= ipl, include_groups=False)
x = raw[raw.match_id.isin(m[m].index) & raw.innings.isin(['1','2'])].copy()
x['season'] = x.delivery_ist_time.str[:4]
runs = x.assign(r=x.runs.astype(int)).groupby(['season','striker_name']).r.sum()
wk = x[x.dismissal_type.isin(['Bowled','Caught','LBW','Stump','Hit Wicket'])].groupby(['season','bowler_name']).size()
pd.DataFrame({'top_run_scorer': runs.groupby(level=0).idxmax().str[1], 'runs': runs.groupby(level=0).max(),
              'top_wicket_taker': wk.groupby(level=0).idxmax().str[1], 'wickets': wk.groupby(level=0).max()})

,top_run_scorer,runs,top_wicket_taker,wickets
season,,,,
2022,Jos Buttler,863,Yuzvendra Chahal,27
2023,Shubman Gill,890,Mohammed Shami,28
2024,Virat Kohli,741,Harshal Patel,24
2025,Sai Sudharsan,759,Prasidh Krishna,25
2026,Vaibhav Sooryavanshi,776,Kagiso Rabada,29


## 4. Win-probability model: out-of-sample accuracy by season
Brier score: lower is better; a coin flip scores 0.25. Each season is scored by a model trained only on earlier seasons.

In [6]:
T = lambda n: pd.read_csv(ROOT / 'outputs' / 'tables' / f'{n}.csv')
T('step1a_wp_eval').round(3)

,season,innings,train_matches,train_seasons,test_matches,test_balls,brier,brier_lo80,brier_hi80,log_loss,coin_flip_brier,base_rate_brier_oracle
0,2023,all,67,2022-2022,71,17307,0.217,0.189,0.249,0.645,0.25,0.250
1,2023,1,67,2022-2022,71,8879,0.248,0.221,0.276,0.701,0.25,0.248
2,2023,2,67,2022-2022,71,8428,0.185,0.150,0.224,0.585,0.25,0.246
3,2024,all,138,2022-2023,70,16902,0.166,0.150,0.186,0.496,0.25,0.250
4,2024,1,138,2022-2023,70,8794,0.205,0.186,0.224,0.595,0.25,0.250
5,2024,2,138,2022-2023,70,8108,0.125,0.100,0.150,0.389,0.25,0.250
6,2025,all,208,2022-2024,69,16555,0.169,0.149,0.188,0.515,0.25,0.246
7,2025,1,208,2022-2024,69,8598,0.205,0.185,0.225,0.604,0.25,0.246
8,2025,2,208,2022-2024,69,7957,0.130,0.107,0.155,0.420,0.25,0.246
9,2026,all,277,2022-2025,71,17116,0.183,0.163,0.203,0.545,0.25,0.246


## 5. Pressure Index: share of balls by level, and example balls

In [7]:
display(T('step1b_bucket_shares').round(3)); T('step1c_example_balls')[['match','situation_before_ball','what_happened','pressure_index','bucket']]

,season,D27_High,D27_Low,D27_Medium,D18strict_High,D18strict_Low,D18strict_Medium,balls
0,2023,0.228,0.466,0.306,0.176,0.437,0.387,17307
1,2024,0.143,0.560,0.297,0.048,0.853,0.099,16902
2,2025,0.224,0.459,0.317,0.086,0.459,0.455,16555
3,2026,0.199,0.514,0.287,0.157,0.489,0.355,17116


,match,situation_before_ball,what_happened,pressure_index,bucket
0,"RCB v CSK, 2026-04-05","CSK 84/4 after 6.4 overs, need 167 off 80",WICKET (Caught),3.0,Low
1,"MI v GT, 2026-04-20",MI 101/4 after 13.4 overs,2 run(s),25.0,Low
2,"CSK v DC, 2026-04-11","DC 156/7 after 16.0 overs, need 57 off 24",1 run(s),45.0,Low
3,"GT v PBKS, 2026-03-31",GT 161/6 after 19.5 overs,1 run(s) [wide],60.0,Medium
4,"SRH v RCB, 2026-05-22",SRH 0/0 after 0.1 overs,2 run(s) [wide],75.0,Medium
5,"PBKS v RR, 2026-04-28","RR 66/1 after 4.2 overs, need 157 off 94",1 run(s),85.0,High
6,"PBKS v MI, 2026-05-14","MI 119/3 after 13.3 overs, need 82 off 39",1 run(s),95.0,High
7,"RR v DC, 2026-05-17","DC 177/5 after 18.3 overs, need 17 off 9",2 run(s) [wide],99.5,High


## 6. The Round 1 answer: top 10 batters and top 10 bowlers under pressure (IPL 2023–26)
Ranked by Clutch Score (0–100 within role) with 80% ranges. The ranges overlap inside each top 10, so read each list as a group, not a strict order.

In [8]:
t10 = json.load(open(ROOT / 'outputs' / 'tables' / 'summary_numbers.json'))['top10']
for r in ['bat', 'bowl']:
    display(pd.DataFrame(t10[r])[['rank','name','team','score','lo','hi','why']].round(1))
    print('#11:', t10['edge'][r])
T('step4_too_close_to_call')

,rank,name,team,score,lo,hi,why
0,1,Vaibhav Sooryavanshi,RR (2026),99.3,91.8,100.0,111 High-pressure balls faced (2023–26): 283 r...
1,2,Ashutosh Sharma,DC (2026),98.7,66.2,100.0,119 High-pressure balls faced (2023–26): 237 r...
2,3,Shreyas Iyer,PBKS (2026),96.1,73.1,99.7,203 High-pressure balls faced (2023–26): 351 r...
3,4,Heinrich Klaasen,SRH (2026),94.3,69.2,99.3,205 High-pressure balls faced (2023–26): 385 r...
4,5,Virat Kohli,RCB (2026),88.1,57.9,98.2,255 High-pressure balls faced (2023–26): 423 r...
5,6,Prabhsimran Singh,PBKS (2026),87.9,44.7,98.0,276 High-pressure balls faced (2023–26): 479 r...
6,7,Marcus Stoinis,PBKS (2026),85.9,37.0,99.3,135 High-pressure balls faced (2023–26): 261 r...
7,8,Sameer Rizvi,DC (2026),80.0,44.9,94.9,101 High-pressure balls faced (2023–26): 165 r...
8,9,Abdul Samad,LSG (2026),79.6,37.5,96.1,112 High-pressure balls faced (2023–26): 200 r...
9,10,Venkatesh Iyer,RCB (2026),79.6,31.0,96.2,106 High-pressure balls faced (2023–26): 193 r...


#11: {'next_name': 'Jos Buttler', 'next_id': '813', 'next_score': 79.244, 'next_lo': 35.719, 'next_hi': 92.839, 'overlaps_10th': True}


,rank,name,team,score,lo,hi,why
0,1,Kuldeep Yadav,DC (2026),99.6,95.5,100.0,226 High-pressure balls bowled (2023–26): 16 w...
1,2,Jasprit Bumrah,MI (2026),98.7,69.4,100.0,209 High-pressure balls bowled (2023–26): 10 w...
2,3,Matheesha Pathirana,KKR (2026),98.4,83.8,100.0,180 High-pressure balls bowled (2023–26): 16 w...
3,4,Patrick Cummins,SRH (2026),96.5,60.5,99.6,123 High-pressure balls bowled (2023–26): 6 wi...
4,5,Ravindra Jadeja,RR (2026),95.6,55.6,99.5,198 High-pressure balls bowled (2023–26): 10 w...
5,6,Krunal Pandya,RCB (2026),93.9,76.3,99.8,170 High-pressure balls bowled (2023–26): 14 w...
6,7,Sunil Narine,KKR (2026),93.7,49.1,99.4,191 High-pressure balls bowled (2023–26): 16 w...
7,8,Vyshak Kumar,PBKS (2026),91.5,36.3,99.0,88 High-pressure balls bowled (2023–26): 4 wic...
8,9,Mohsin Khan,LSG (2026),89.7,38.7,99.7,113 High-pressure balls bowled (2023–26): 4 wi...
9,10,Lungi Ngidi,DC (2026),87.3,67.5,95.9,63 High-pressure balls bowled (2023–26): 6 wic...


#11: {'next_name': 'Jofra Archer', 'next_id': '3298', 'next_score': 85.492, 'next_lo': 66.868, 'next_hi': 98.244, 'overlaps_10th': True}


,role,top10_pairs,pairs_separated_by_80pct_ranges,pairs_too_close_to_call
0,bat,45,0,45
1,bowl,45,0,45


## 6b. Does the pre-registered scope (players active in IPL 2025 or 2026) change the top 10s?
Re-scored with every qualifying 2023–26 player, same frozen method.

In [9]:
T('step4_scope_sensitivity')

,role,rank_all_players,player_name,score_all_players,last_ipl_season,in_headline_top10,n_rated_all,n_rated_headline,overlap_top10,entering,leaving
0,bat,1.0,Vaibhav Sooryavanshi,99.33,2026.0,True,NaN,NaN,NaN,NaN,NaN
1,bat,2.0,Ashutosh Sharma,98.68,2026.0,True,NaN,NaN,NaN,NaN,NaN
2,bat,3.0,Shreyas Iyer,96.08,2026.0,True,NaN,NaN,NaN,NaN,NaN
3,bat,4.0,Heinrich Klaasen,94.29,2026.0,True,NaN,NaN,NaN,NaN,NaN
4,bat,5.0,Virat Kohli,88.11,2026.0,True,NaN,NaN,NaN,NaN,NaN
5,bat,6.0,Prabhsimran Singh,87.88,2026.0,True,NaN,NaN,NaN,NaN,NaN
6,bat,7.0,Marcus Stoinis,85.86,2026.0,True,NaN,NaN,NaN,NaN,NaN
7,bat,8.0,Sameer Rizvi,79.99,2026.0,True,NaN,NaN,NaN,NaN,NaN
8,bat,9.0,Abdul Samad,79.62,2026.0,True,NaN,NaN,NaN,NaN,NaN
9,bat,10.0,Venkatesh Iyer,79.57,2026.0,True,NaN,NaN,NaN,NaN,NaN


## 7. Backtest: does the score persist into the next season?
The score is built from seasons before N and checked in season N. It is compared with a naive ranking and with a random pick.

In [10]:
T('step4_backtest_summary')[['role','fold','score_hit_rate_excl','naive_hit_rate_excl','base_rate_random_pick','hit_diff_lo80','hit_diff_hi80','claim']].round(1)

,role,fold,score_hit_rate_excl,naive_hit_rate_excl,base_rate_random_pick,hit_diff_lo80,hit_diff_hi80,claim
0,bat,2024,75.0,60.0,66.7,NaN,NaN,NaN
1,bat,2025,16.7,50.0,43.5,NaN,NaN,NaN
2,bat,2026,50.0,25.0,53.3,NaN,NaN,NaN
3,bat,pooled,42.9,46.7,50.8,-17.1,3.8,historical indicator; persistence not demonstr...
4,bowl,2024,50.0,50.0,66.7,NaN,NaN,NaN
5,bowl,2025,85.7,70.0,54.5,NaN,NaN,NaN
6,bowl,2026,79.5,100.0,60.9,NaN,NaN,NaN
7,bowl,pooled,72.8,75.0,59.5,-14.6,4.4,historical indicator; persistence not demonstr...


## 8. Robustness and sensitivity

In [11]:
display(T('step4_definition_robustness').round(2)); T('step4_sensitivity').groupby(['role','component']).top10_overlap_pct.min().unstack(0)

,variant,role,scorable_main,scorable_variant,top20_overlap_pct,top10_overlap_pct,spearman_common
0,leverage_only,bat,47,45,70.0,40.0,0.83
1,leverage_only,bowl,71,70,80.0,60.0,0.81
2,rule_based,bat,47,40,20.0,20.0,0.21
3,rule_based,bowl,71,73,40.0,30.0,0.41
4,with_close_game,bat,47,48,85.0,70.0,0.96
5,with_close_game,bowl,71,73,95.0,90.0,0.97
6,d18_strict_buckets,bat,47,21,40.0,40.0,0.81
7,d18_strict_buckets,bowl,71,50,65.0,60.0,0.75
8,season_centred,bat,47,47,100.0,100.0,1.00
9,season_centred,bowl,71,71,100.0,100.0,0.99


role,bat,bowl
component,,
C,90.0,100.0
H,100.0,100.0
I,100.0,100.0
S,90.0,100.0
consistency_only,40.0,60.0
equal_weights,90.0,90.0
skill_only,80.0,80.0


## 9. Players in scope but not rated (sample below the pre-registered minimum)
Shown so nobody is silently dropped: at least 100 High-pressure balls faced (batters) or 60 legal High-pressure balls bowled (bowlers) in 2023–26 are needed.

In [12]:
T('step5_not_scorable').head(15)

,player_id,player_name,hi_balls_faced,hi_legal_bowled,reason
0,1205,Jonny Bairstow,73.0,0.0,Below minimum High-pressure sample (bat ≥100 f...
1,1440,Rahul Tripathi,62.0,0.0,Below minimum High-pressure sample (bat ≥100 f...
2,1487,Adam Milne,2.0,13.0,Below minimum High-pressure sample (bat ≥100 f...
3,1556,David Payne,5.0,3.0,Below minimum High-pressure sample (bat ≥100 f...
4,1596,Moeen Ali,17.0,57.0,Below minimum High-pressure sample (bat ≥100 f...
5,19312,Nuwan Thushara,0.0,37.0,Below minimum High-pressure sample (bat ≥100 f...
6,1941,Devon Conway,94.0,0.0,Below minimum High-pressure sample (bat ≥100 f...
7,19714,Abishek Porel,59.0,0.0,Below minimum High-pressure sample (bat ≥100 f...
8,19781,Mitchell Owen,0.0,0.0,Below minimum High-pressure sample (bat ≥100 f...
9,198,Manish Pandey,53.0,0.0,Below minimum High-pressure sample (bat ≥100 f...


## 10. Reproducibility check
Re-running gives identical tables: the script is deterministic, with seed 2027. The checksums of the key outputs are printed below.

In [13]:
import hashlib
for f in ['step4_scores_batters','step4_scores_bowlers','step4_backtest_summary','step4_scope_sensitivity']:
    print(f, hashlib.sha256((ROOT/'outputs'/'tables'/f'{f}.csv').read_bytes()).hexdigest()[:16])

step4_scores_batters 2a37861a75eb1455
step4_scores_bowlers fbdaab6ab1d6109a
step4_backtest_summary 126a8899c4064939
step4_scope_sensitivity 25d7348056778793
